In [0]:
display(dbutils.fs.ls("s3://vendor-status-monitor/raw/status_checks/"))

In [0]:
display(dbutils.fs.ls("s3://vendor-status-monitor/raw/status_checks/vendor=digitalocean/"))

In [0]:
df = spark.read.json("s3://vendor-status-monitor/raw/status_checks/vendor=digitalocean/dt=2026-06-24/*.json")
display(df)

In [0]:
df.printSchema()

In [0]:
test_df = spark.read.schema(status_page_schema).json(
    "s3://vendor-status-monitor/raw/status_checks/*/*/*.json"
)

display(test_df)

In [0]:
test_df.count()

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, LongType, ArrayType
)

incident_update_schema = StructType([
    StructField("id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("body", StringType(), True),
    StructField("created_at", StringType(), True),
])

incident_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("impact", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("incident_updates", ArrayType(incident_update_schema), True),
])

component_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("group", BooleanType(), True),
    StructField("group_id", StringType(), True),
    StructField("position", LongType(), True),
    StructField("description", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
])

page_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("url", StringType(), True),
    StructField("time_zone", StringType(), True),
    StructField("updated_at", StringType(), True),
])

status_schema = StructType([
    StructField("indicator", StringType(), True),
    StructField("description", StringType(), True),
])

status_page_schema = StructType([
    StructField("_ingested_at", StringType(), True),
    StructField("page", page_schema, True),
    StructField("status", status_schema, True),
    StructField("components", ArrayType(component_schema), True),
    StructField("incidents", ArrayType(incident_schema), True),
    StructField("scheduled_maintenances", ArrayType(incident_schema), True),
])

In [0]:
%sql
SELECT COUNT(*) FROM vendor_status.bronze.status_checks;

In [0]:
%sql
SELECT page.name AS vendor, COUNT(*) AS row_count
FROM vendor_status.bronze.status_checks
GROUP BY page.name
ORDER BY row_count DESC;

In [0]:
%sql
SELECT * FROM vendor_status.bronze.status_checks LIMIT 5;

In [0]:
%sql
SELECT
  page.name AS vendor,
  status.indicator,
  status.description,
  size(components) AS num_components,
  size(incidents) AS num_incidents,
  size(scheduled_maintenances) AS num_scheduled_maintenances,
  _ingested_at,
  _bronze_loaded_at
FROM vendor_status.bronze.status_checks
ORDER BY _bronze_loaded_at DESC
LIMIT 20;